# Market Sentiment Analysis of Guardian Business Articles

## Overview
This project investigates whether machine learning models can reliably classify 
the sentiment of financial news articles from *The Guardian* on the scale below:
- 4 - POSITIVE
- 3 - LEANING POSITIVE
- 2 - NEUTRAL
- 1 - LEANING NEGATIVE
- 0 - NEGATIVE

## Models
We used four models over the course of this project evaluated on ground truth labels generated by ensembling two LLMs — **ZotGPT** and **Gemini** — via their respective APIs, averaging their predictions per article.

- Logistic Regression + TF-IDF - Baseline
- FinBERT (zero-shot) - Pretrained transformer baseline 
- FinBERT (fine-tuned) - Using hugging-face transformers
- LLaMA Zero-Shot and Few-Shot Baseline Method

This notebook will go over a simple pipeline on a sample dataset of articles using the baseline FinBert model as inference, including data proprocessing, model evaluation, and stock market correlation.


In [3]:
import json 
import os 
import pandas as pd 
from transformers import pipeline
from sklearn.metrics import mean_absolute_error, accuracy_score, classification_report
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

## LLM Committee Labeling

Each article's title and first 2,000 characters of body text were extracted and formatted into a structured prompt for LLM labeling. Let's take a look at what a prompt that goes into the API call might look like.

In [9]:
with open("../data/raw/sample_guardian_articles.json", encoding="utf-8") as f:
    articles = json.load(f)
    
def create_sentiment_prompt(article):
    title = article.get("webTitle", "")
    body = article.get("fields", {}).get("bodyText", "")

    text_to_analyze = f"{title}\n\n{body[:2000]}"

    prompt = f"""You are a financial sentiment analysis expert. Analyze the following Guardian business article and classify its sentiment regarding the US stock market.

Article:
{text_to_analyze}

Classify the sentiment into one of these categories:
4 - POSITIVE: Optimistic outlook, growth, gains, bullish sentiment, or favorable conditions for US stocks
3 - LEANING POSITIVE: Mostly positive but with some caveats or qualifiers
2 - NEUTRAL: Balanced view, mixed signals, or not directly related to US stock market sentiment
1 - LEANING NEGATIVE: Mostly negative but with some caveats or qualifiers
0 - NEGATIVE: Pessimistic outlook, losses, bearish sentiment, concerns, or unfavorable conditions for US stocks

Respond with ONLY the number (0, 1, 2, 3, or 4) and nothing else."""

    return prompt

print(create_sentiment_prompt(articles[0]))

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/sample_guardian_articles.json'

## Building the Dataset

The output labels from the prompt input into ZotGPT and Gemini were then averaged and stored in new JSON object. From here, we'll create a label map mapping each label to an article ID. This dictionary was then used to map an article's important text to its id, producing a parallel texts and labeled list. We will then have two matching arrays of texts and labels ready for training.

In [7]:
def load_averaged_labels(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    label_map = {}
    for id, score in data.items():
        label_map[id] = int(score)

    return label_map

def build_dataset_from_labels(articles, label_map):
    texts = []
    labels = []

    for art in articles:
        headline = art["fields"]["headline"]
        standfirst = art["fields"]["standfirst"]
        body = art["fields"]["bodyText"]

        text = headline + " " + standfirst + " " + body

        texts.append(text)
        labels.append(label_map[art["id"]])

    return texts, labels

averaged_labels = load_averaged_labels("../data/processed/sample_averaged_labels.json")
texts_eval, y_eval = build_dataset_from_labels(articles, averaged_labels)


KeyError: 'business/live/2026/mar/13/uk-gdp-report-economy-growing-iran-energy-shock-rachel-reeves-us-consumer-confidence-news-updates'

## FinBERT Baseline (Zero-Shot)

This zero-shot FinBERT will serve as a strong pretrained baseline. Since FinBERT natively outputs three classes (positive,
neutral, negative), predictions were mapped to our 0–4 scale using confidence thresholding:

In [17]:
def run_finbert(texts):
    finbert = pipeline(
        "text-classification",
        model="ProsusAI/finbert"
    )
    y_pred = []
    for text in texts:
        result = finbert(
            text,
            truncation=True,
            max_length=512
        )[0]

        label = result["label"].lower()
        score = result["score"]

        if label == "negative":
            if score > 0.75:
                y_pred.append(0)
            else:
                y_pred.append(1)

        elif label == "neutral":
            y_pred.append(2)

        elif label == "positive":
            if score > 0.75:
                y_pred.append(4)
            else:
                y_pred.append(3)

    return y_pred

In [18]:
def evaluate_model(y_true, y_pred):
    """ 
    Runs all evaluation metrics and prints results.
    """

    mae = mean_absolute_error(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)

    print("\n" + "=" * 60)
    print("=" * 60)

    print(f"Accuracy: {acc:.3f}")
    print(f"MAE: {mae:.3f}")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

evaluate_model(y_eval, run_finbert(texts_eval))

[2, 2, 4, 2, 0, 3, 4, 4, 3, 0, 2, 0, 0, 0, 0, 0, 0, 2, 4, 2]


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 564.49it/s, Materializing param=classifier.weight]                                      
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Accuracy: 0.600
MAE: 0.750

Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.88      0.78         8
           1       0.00      0.00      0.00         0
           2       0.50      0.33      0.40         6
           3       1.00      0.50      0.67         2
           4       0.67      0.50      0.57         4

    accuracy                           0.60        20
   macro avg       0.57      0.44      0.48        20
weighted avg       0.66      0.60      0.61        20



c:\Users\robbi\source\repos\CS175-Market-Sentiment\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\robbi\source\repos\CS175-Market-Sentiment\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\robbi\source\repos\CS175-Market-Sentiment\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"

## Stock Market Correlation Analysis

To analyze if news sentiment can predict stock market data, we built a script to test their alignment over the course of ~2 years. Smoothing on a five-day rolling basis was applied to account for noise from daily volatility of stock data. This script generates a dual time-series graph of news sentiment and market data.

In [ ]:
with open('data/processed/averaged_labels.json', 'r') as f:
    sentiment_data = json.load(f)

month_map = {
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
    'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
}

KeyboardInterrupt: 

In [ ]:
def extract_date(article_id):
    """
    Extract date from article ID.
    Article IDs are in format: business/YYYY/mon/DD/title
    """
    parts = article_id.split('/')
    if len(parts) < 4:
        return None
    
    try:
        year = int(parts[1])
        month_str = parts[2].lower()
        if month_str not in month_map:
            return None
        month = month_map[month_str]
        day = int(parts[3])
        
        date = datetime(year, month, day)
        return date.date()
    except (ValueError, IndexError):
        return None

In [ ]:
# combine sentiment data
sentiment_by_date = {}
for article_id, sentiment in sentiment_data.items():
    date = extract_date(article_id)
    if date:
        if date not in sentiment_by_date:
            sentiment_by_date[date] = []
        sentiment_by_date[date].append(sentiment)

avg_sentiment = {date: sum(sentiments) / len(sentiments) for date, sentiments in sentiment_by_date.items()}

sentiment_df = pd.DataFrame(
    list(avg_sentiment.items()),
    columns=['Date', 'Sentiment']
)

sentiment_df['Date'] = pd.to_datetime(sentiment_df['Date'])
sentiment_df.set_index('Date', inplace=True)

# load stock data
stock_df = pd.read_csv('results/stock_results.csv', sep=',', index_col=0, parse_dates=True)

# compute daily returns
stock_df['Return'] = stock_df['Close'].pct_change()

# normalize index to date only (remove time and timezone)
stock_df.index = pd.to_datetime(stock_df.index, utc=True)
stock_df['Date'] = stock_df.index.date
stock_df.set_index('Date', inplace=True)

combined_df = sentiment_df.join(stock_df[['Return']], how='inner')
combined_df = combined_df.sort_index()

# smooth signals, 5-day roll
combined_df['Return_Smoothed'] = combined_df['Return'].rolling(5).mean()
combined_df['Sentiment_Smoothed'] = combined_df['Sentiment'].rolling(5).mean()

if not combined_df.empty:
    print(combined_df.head())

    # plot dual time series
    fig, ax1 = plt.subplots(figsize=(12,6))
    ax1.plot(
        combined_df.index,
        combined_df['Return_Smoothed'],
        color='blue',
        label='Stock Return',
        alpha=0.7
    )

    ax1.set_ylabel("Stock Return", color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')

    # sentiment axis
    ax2 = ax1.twinx()
    ax2.plot(
        combined_df.index,
        combined_df['Sentiment_Smoothed'],
        color='red',
        label='Sentiment Score',
        alpha=0.7
    )
    ax2.set_ylabel("Average Sentiment", color='red')
    ax2.tick_params(axis='y', labelcolor='red')

    plt.title("Market Sentiment vs Stock Returns Over Time")
    plt.grid(True, alpha=0.3)

    plt.savefig(
        'results/sentiment_stock_timeseries.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.show()

    # lagged sentiment analysis: does news sentiment predict the stock market?
    lags = range(1,6)
    lag_corrs = []

    for lag in lags:
        corr = combined_df['Sentiment'].corr(combined_df['Return'].shift(-lag))
        lag_corrs.append(corr)

    plt.figure(figsize=(8,5))
    plt.bar(lags, lag_corrs)
    plt.xticks(lags)
    plt.axhline(0, color='black', linestyle='--', alpha=0.5)
    plt.xlabel("Days Ahead")
    plt.ylabel("Correlation")
    plt.title("Sentiment vs Future Returns")

    plt.savefig(
        'results/lagged_sentiment_analysis.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.show()
    
else:
    print("No overlapping dates found between sentiment data and stock data.")